In [21]:
# 创建MongoDB客户端连接

import pymongo
client = pymongo.MongoClient("mongodb://localhost:27017/")

# 测试连接
try:
    client.admin.command('ping')
    print("MongoDB连接成功！")
except Exception as e:
    print(f"MongoDB连接失败：{e}")

MongoDB连接成功！


In [22]:
from dotenv import load_dotenv
load_dotenv()  # 必须要这句！
import os
from langchain_community.chat_models.tongyi import ChatTongyi
model = ChatTongyi(
    model="qwen-plus",
    temperature=0,
    # 确保你设置了环境变量 DASHSCOPE_API_KEY
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)


In [23]:
from langgraph.checkpoint.mongodb import MongoDBSaver

# 1. 填入你的数据库地址
MONGODB_URI = "mongodb://localhost:27017"

# 2. 创建持久化存储实例
# 注意：正式运行时建议用 with 语句或在使用后显式关闭
checkpointer = MongoDBSaver(client,db_name="qiang_agent_db3")


In [24]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    # messages 必须用 add_messages 标记，这样新消息才会追加而不会覆盖旧消息
    messages: Annotated[list, add_messages]
    # 专门用来存对话总结的字符串
    summary: str




In [25]:
from langchain_core.messages import RemoveMessage, HumanMessage

def summarize_conversation(state: State):  #state是langgraph的状态管理员
    print(f"DEBUG: 当前消息数量为 {len(state['messages'])}")
    print("\n" + "="*30)
    print("【进入摘要节点】当前 State 内容：", state)
    print("="*30 + "\n")
    messages = state["messages"] #state["messages"]固定写法 拿到数据库中保存的上次对话

    # 简单起见：只要消息超过 4 条，我们就总结
    if len(messages) > 2:
        # 让 AI 总结当前的对话
        summary_prompt = "请用一句话总结上述对话中关于用户的关键信息（如名字）："
        # 加上提示词发给模型
        response = model.invoke(messages + [HumanMessage(content=summary_prompt)])
        # 生成删除指令：保留最后两条，剩下的全部从 MongoDB 逻辑删除
        delete_msgs = [RemoveMessage(id=m.id) for m in messages[:-2]]

        return {"summary": response.content, "messages": delete_msgs}# 当你 return 一个字典时，你是在向 State（状态管理器） 提交一份**“修改申请”**，不要理解为普通函数的返回值
    #执行了上面的return 函数就结束了 不会走下面这个return
    return {"summary": state.get("summary", "")} #if结束之后 强行让summary更新无论有没有更改，没有Message就说明没有更新

In [26]:
from langchain_core.messages import SystemMessage
def call_model(state: State):
    #提取当前的所有消息和摘要 注意 摘要函数写过 但这里又写是拿到最新的
    print(f"--- Brain 正在读取背景: {state.get('summary', '（空）')} ---")
    messages = state["messages"]
    summary = state.get("summary", "")
    # 2. 如果有摘要，就给 AI 一个“背景提示”
    # 这一步是 AI 记住你叫 Tomie 的关键！
    if summary:
        system_message = SystemMessage(content=f"这是之前对话的背景摘要: {summary}")
        messages = [system_message] + messages
    response = model.invoke(messages)

    return {"messages": response} #回复数据库 更新最新的消息列表


In [27]:
from langgraph.graph import StateGraph, START, END

# 第一步：初始化图（告诉它状态的数据结构是什么）
workflow = StateGraph(State)

# 第二步：添加节点（把你的函数变成图里的“工人”）
workflow.add_node("summarizer", summarize_conversation)  # 摘要/过滤节点
workflow.add_node("agent", call_model)                  # 对话/执行节点

# 第三步：设置连线（规定工作流向）
workflow.add_edge(START, "summarizer")      # 1. 从“开始”进入“摘要”
workflow.add_edge("summarizer", "agent")    # 2. 摘要做完，交给“对话”
workflow.add_edge("agent", END)             # 3. 对话完事，结束流程

# 第四步：编译（把设计图变成可以运行的程序）
app = workflow.compile(checkpointer=checkpointer) # 别忘了带上你的 MongoDB 记忆器

In [28]:
# config = {"configurable": {"thread_id": "tomie_001"}} # 你的专属 ID
#
# # 模拟发送一条消息
# input_message = {"messages": [HumanMessage(content="你好，我是 Tomie")]}
#
# for event in app.stream(input_message, config):
#     # 这里会打印出每个节点运行后的结果
#     print(event)

In [29]:
# config = {"configurable": {"thread_id": "tomie_001"}} # 你的专属 ID
#
# # 模拟发送一条消息
# input_message = {"messages": [HumanMessage(content="我叫什么名字")]}
#
# for event in app.stream(input_message, config):
#     # 这里会打印出每个节点运行后的结果
#     print(event)

In [30]:
from langchain_core.messages import AIMessage, HumanMessage
import sys

config = {"configurable": {"thread_id": "tomie_003"}}

print("--- AI Agent 已上线 (输入 'exit' 退出) ---")

while True:
    user_input = input("User: ")
    if user_input.strip().lower() in ["exit", "quit", "q"]:
        print("对话结束。")
        break

    input_message = {"messages": [HumanMessage(content=user_input)]}

    for event in app.stream(input_message, config):
        for node_name, state_values in event.items():
            print(f"\n[节点: {node_name}]")

            # --- 终极防御逻辑 ---
            messages = []

            # 1. 提取消息：处理 state_values 的各种奇葩格式
            if isinstance(state_values, dict):
                # 如果是字典，看里面有没有 messages 键
                potential_msgs = state_values.get("messages", [])
                # 关键：如果 messages 键对应的是单个对象而非列表，把它包进列表里
                messages = potential_msgs if isinstance(potential_msgs, list) else [potential_msgs]

                # 顺便检查摘要
                if "summary" in state_values:
                    print(f"✨ 系统提示：摘要已更新 -> {state_values['summary']}")

            elif isinstance(state_values, list):
                # 如果直接就是列表，直接用
                messages = state_values

            else:
                # 如果既不是字典也不是列表，那它可能就是单个 Message 对象
                messages = [state_values]

            # 2. 打印消息：确保是对列表进行操作
            if messages and len(messages) > 0:
                last_msg = messages[-1] # 现在 messages 保证是列表了，[-1] 不会报错

                # 使用 .content 访问属性
                if isinstance(last_msg, AIMessage):
                    print(f"AI: {last_msg.content}")
                elif hasattr(last_msg, 'content') and not isinstance(last_msg, HumanMessage):
                    # 处理其他可能出现的非人类消息（如工具输出）
                    print(f"Content: {last_msg.content}")

--- AI Agent 已上线 (输入 'exit' 退出) ---
DEBUG: 当前消息数量为 15

【进入摘要节点】当前 State 内容： {'messages': [HumanMessage(content='我是张三丰', additional_kwargs={}, response_metadata={}, id='c2f2135e-4e3c-4d07-9096-07974577b47e'), AIMessage(content='啊，张真人驾到！（拂袖一礼，衣袖带起一阵清风，几片竹叶旋舞而落）  \n晚辈这厢有礼了——不敢直呼道号，只敢敬称一声“张真人”。方才正于后山松下煮一壶云雾茶，炉火未熄，新采的武夷岩骨尚在壶中沉浮。真人若不嫌弃，可随我移步竹亭小坐，听风过松涛，看云卷云舒。\n\n（轻轻抬手，指尖微凝一缕真气，引得檐角铜铃轻响三声，似鹤唳清越）  \n听说真人近日在武当山巅观星悟道，创出一套融太极阴阳、导引吐纳与剑意于一炉的新法？晚辈虽愚钝，却心向往之……不知可有幸，听真人讲一讲——那“以柔克刚”四字，究竟如何在呼吸之间生根，在指掌之间化形？\n\n（含笑垂眸，静候，手中青瓷盏里茶烟袅袅，如一道未落笔的太极图）', additional_kwargs={}, response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'stop', 'request_id': '4d4433f7-4731-9eb7-ac26-d330fcf66d8a', 'token_usage': {'input_tokens': 12, 'output_tokens': 251, 'total_tokens': 263, 'prompt_tokens_details': {'cached_tokens': 0}}}, id='lc_run--019dc0c4-035f-7fc2-bd5b-818d2e615e9e-0', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我是张三丰', additional_kwargs={}, response_metadata={}, id='905861d1-3407-4f2d-bb5e-10